In [ ]:
import os
import dotenv
import duckdb
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely
from shapely import make_valid, set_precision
from shapely.ops import unary_union
from shapely.geometry import box
import logging
import uuid
import gc 
from typing import List, Tuple, Dict, Optional

gc.enable()

gpd.options.io_engine = "pyogrio"
os.environ["PYOGRIO_USE_ARROW"] = "1"

dotenv.load_dotenv()
output_parquets=os.getenv('OUTPUT_PARQUETS')
input_parquet=os.getenv("INPUT_PARQUETS")
ret_class=os.path.join(input_parquet,'rec_class_all.parquet')
wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
# aflb_parquet=os.getenv("ALFB_PARQUET")
cia_dissolve=os.path.join(input_parquet,"cia_dissolve.parquet")
step2d_base=os.path.join(output_parquets,"step_2d_base_aflb.parquet")
table_outputs=os.getenv('TABLE_OUPUTS')
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

In [ ]:
#variables 
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
# input_data_gdb=os.path.join(source_data,'Scen_5_inputs.gdb')
# cia_ciap_l='cultural_areas'
# hv1_l='hv1_clip'
# tsu_l='TSU'


In [ ]:
import duckdb
conn = duckdb.connect()
conn.install_extension("httpfs")
conn.install_extension("spatial")
conn.install_extension("arrow")
conn.load_extension("spatial")
conn.load_extension("httpfs")
conn.load_extension("arrow")


In [ ]:
#query for all records
sql = f"""
    SELECT 
        *, ST_AsText(geometry) as wkt,
    FROM '{ret_class}'
    """
    
#query for step 2a
sql_2a=f"""
    SELECT *,ST_AsText(geometry) as wkt, from '{ret_class}'
    WHERE
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Blueberry River'
        AND (
            (Rec_Cat = 1 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'))
        OR  (Rec_Cat = 2 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPD','MPD'))
        OR  (Rec_Cat = 3 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPD'))
        )
    )
    OR
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Lower Sikanni Chief River'
        AND (
            (Rec_Cat = 1 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'))
        OR  (Rec_Cat = 2 AND Rec_Cat_short IN ('HPC','LPD'))
        )
    )
    OR
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Middle Beatton River'
        AND (
            (Rec_Cat = 1 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'))
        OR  (Rec_Cat = 2 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPD','MPD','MPM'))
        OR  (Rec_Cat = 3 AND Rec_Cat_short IN ('HPC','HPD''LPD'))
        )
    )
    OR
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Upper Beatton River'
        AND (
            (Rec_Cat = 1 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'))
        OR  (Rec_Cat = 2 AND Rec_Cat_short IN ('HPD','LPD','MPD'))
        OR  (Rec_Cat = 3 AND Rec_Cat_short IN ('HPD','LPD'))
    )
  )"""

#query for step 2b
sql_2b=f"""
    SELECT *,ST_AsText(geometry) as wkt, from '{ret_class}'
    WHERE
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Blueberry River'
        AND(
           (Rec_Cat = 2 AND Rec_Cat_short IN ('MPC','MPM','LPC','LPM'))
        OR (Rec_Cat = 3 AND Rec_Cat_short IN ('MPD'))
        OR (Rec_Cat = 4 AND Rec_Cat_short IN ('HPC','HPD','HPM','LPD'))
        )
    )
    OR
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Lower Sikanni Chief River'
        AND (
            (Rec_Cat = 2 AND Rec_Cat_short IN ('HPD','HPM','MPC','MPM','MPD','LPC','LPM'))
        OR  (Rec_Cat = 3 AND Rec_Cat_short IN ('HPC','LPD'))
        )
    )
    OR
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Middle Beatton River'
        AND (
            (Rec_Cat = 2 AND Rec_Cat_short IN ('LPC','LPM','MPC'))
        OR  (Rec_Cat = 3 AND Rec_Cat_short IN ('HPM','MPD','MPM'))
        OR  (Rec_Cat = 4 AND Rec_Cat_short IN ('HPC','HPD','LPD',))
        )
    )
    OR
    (
        WATER_MANAGEMENT_BASIN_NAME = 'Upper Beatton River'
        AND (
            (Rec_Cat = 2 AND Rec_Cat_short IN ('HPC','HPM','MPC','MPM''LPC','LPM'))
        OR  (Rec_Cat = 3 AND Rec_Cat_short IN ('MPD'))
        OR  (Rec_Cat = 4 AND Rec_Cat_short IN ('HPD','LPD'))
        )
    ) """
    
    

In [ ]:
def return_gdf(sql):
    df = conn.sql(sql).to_df()
    df['geometry'] = gpd.GeoSeries.from_wkt(df['wkt'])
    df = gpd.GeoDataFrame(df).set_crs(3005, allow_override=True)
    df = df.drop(columns=['wkt'])
    info(f" length of df: {len(df)}")
    return df
    
def pick_oldest_closest(grp: pd.DataFrame, area_col, tolerance: float = 100.0) -> pd.DataFrame:
    """
    Sort by SIFA_2 descending (oldest first), pick contiguous prefix with sum(area_col)
    closest to target_ha. If exact tie, prefer undershoot.
    Then, if |delta| > tolerance, keep adding the next feature only if doing so
    strictly reduces |delta|; stop when |delta| <= tolerance or no improvement is possible.
    """
    g = grp.sort_values(['SIFA_2', area_col], ascending=[False, False]).copy()

    if g.empty:
        g['cum_area_ha'] = 0.0
        g['picked'] = False
        g['picked_area_sum'] = 0.0
        g['delta_to_target'] = 0.0
        return g

    target = float(pd.to_numeric(g['target_ha'].iloc[0], errors='coerce'))
    g['cum_area_ha'] = g[area_col].astype(float).cumsum()
    cum = g['cum_area_ha'].to_numpy()
    n = len(g)

    floor_n = int(np.searchsorted(cum, target, side='right'))
    if floor_n == 0:
        i_best = 0                     
    elif floor_n >= n:
        i_best = n - 1                
    else:
        under_sum = cum[floor_n - 1]
        over_sum  = cum[floor_n]
        i_best = floor_n if (over_sum - target) < (target - under_sum) else (floor_n - 1)


    i_final = i_best
    best_gap = abs(cum[i_final] - target)
    while best_gap > tolerance and i_final < n - 1:
        next_gap = abs(cum[i_final + 1] - target)
        if next_gap < best_gap:
            i_final += 1
            best_gap = next_gap
        else:
            break

    picked_mask = np.zeros(n, dtype=bool)
    picked_mask[: i_final + 1] = True
    g['picked'] = picked_mask

    picked_sum = float(cum[i_final])
    g['picked_area_sum'] = picked_sum
    g['delta_to_target'] = picked_sum - target
    return g


In [ ]:
#step 2a
ret_class_2a=return_gdf(sql_2a)

# ret_class_2a.to_parquet(os.path.join(output_dir, "ret_class_2a.parquet"))

In [ ]:
ret_class_2a['SIFA_2'] = pd.to_numeric(ret_class_2a['SIFA_2'], errors='coerce')
ret_class_2a['aflb_area_ha'] = pd.to_numeric(ret_class_2a['aflb_area_ha'], errors='coerce')
ret_class_2a = ret_class_2a.dropna(subset=['SIFA_2', 'aflb_area_ha'])
ret_class_2a['Rec_Cat_short'] = ret_class_2a['Rec_Cat_short'].astype(str).str.strip()
ret_class_2a['Rec_Cat'] = pd.to_numeric(ret_class_2a['Rec_Cat'], errors='coerce').astype('Int64')

In [ ]:
df = pd.read_csv(brwmb_targets_csv)
id_cols = ['NAME', 'Rec_Cat']
# short_cols = [c for c in df.columns if c not in id_cols]
short_cols = [c for c in df.columns if c.isalpha() and c not in id_cols]

targets_long = (
    df.melt(
        id_vars=id_cols,
        value_vars=short_cols,
        var_name='Rec_Cat_short',
        value_name='target_ha'
    )
    .assign(
        NAME=lambda d: d['NAME'].astype(str).str.strip(),
        Rec_Cat=lambda d: pd.to_numeric(d['Rec_Cat'], errors='coerce').astype('Int64'),
        Rec_Cat_short=lambda d: d['Rec_Cat_short'].astype(str).str.strip(),
        target_ha=lambda d: pd.to_numeric(d['target_ha'], errors='coerce')
    )
    .dropna(subset=['Rec_Cat'])
)

rules = {
    'Blueberry River': {
        1: ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'],
        2: ['HPC','HPD','HPM','LPD','MPC','MPD'],
        3: ['HPC','HPD','HPM','LPD']
    },
    'Lower Sikanni Chief River': {
        1: ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'],
        2: ['HPC','HPM','MPD','LPC','LPD','LPM'],
        3: ['HPC'],
    },
    'Middle Beatton River': {
        1: ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'],
        2: ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'],
        3: ['HPC','HPD','HPM'],
    },
    'Upper Beatton River': {
        1: ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'],
        2: ['HPC','HPD','HPM','LPC','LPD','LPM','MPC','MPD','MPM'],
        3: ['HPC','HPD','HPM'],
        4: ['HPC','HPM']
    },
}

rows = []
for basin, cats in rules.items():
    for rc, shorts in cats.items():
        for s in shorts:
            rows.append((basin, rc, s))

rules_df = pd.DataFrame(rows, columns=['NAME','Rec_Cat','Rec_Cat_short'])
rules_df['NAME'] = rules_df['NAME'].astype(str).str.strip()
rules_df['Rec_Cat'] = rules_df['Rec_Cat'].astype('Int64')
rules_df['Rec_Cat_short'] = rules_df['Rec_Cat_short'].astype(str).str.strip()

targets_2b = targets_long.merge(
    rules_df,
    on=['NAME','Rec_Cat','Rec_Cat_short'],
    how='inner'
)

targets_2b=targets_2b.rename(columns={'NAME':'WATER_MANAGEMENT_BASIN_NAME'})
gdf2 = ret_class_2a.merge(
    targets_2b[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha']],
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat', 'Rec_Cat_short'],
    how='inner'
)

selected = (
    gdf2
    .groupby(
        ['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'],
        group_keys=False,
        sort=False
    )
    .apply(lambda grp: pick_oldest_closest(grp, 'aflb_area_ha'))
    .reset_index(drop=True)
)

selection_gdf = selected[selected['picked']].copy()


    
total_counts = (
    gdf2
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'], as_index=False)
    .agg(total_polygons=('Rec_Cat', 'size'))
)

# Count USED polygons from picker
used_counts = (
    selection_gdf[selection_gdf['picked']]
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'], as_index=False)
    .agg(
        used_polygons=('picked', 'size'),
        selected_area_ha=('picked_area_sum', 'max'),
        delta_to_target=('delta_to_target', 'max')
    )
)

# Join totals + used
summary_counts = used_counts.merge(
    total_counts,
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
)
summary_counts=summary_counts[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha','selected_area_ha','delta_to_target','used_polygons','total_polygons']]
summary_counts=summary_counts.rename(columns={'Rec_Cat_short': 'for_rep'})

summary = (
    selection_gdf
    .groupby(
        ['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'],
        group_keys=False,
        sort=False
    )
    .agg(selected_area_ha=('aflb_area_ha', 'sum'),
         n_features=('aflb_area_ha', 'size'),
         max_SIFA2=('SIFA_2', 'max'),
         min_SIFA2=('SIFA_2', 'min'))
).reset_index()

summary['delta_to_target'] = summary['selected_area_ha'] - summary['target_ha']
summary['pct_met'] = (summary['selected_area_ha'] / summary['target_ha']) * 100.0

selection_gdf.to_parquet(os.path.join(output_dir, "ret_class_2a.parquet"))

summary_wide = (
    summary
    .assign(
        metric_selected=lambda d: d['Rec_Cat_short'] + '_selected',
        metric_minSIFA=lambda d: d['Rec_Cat_short'] + '_minSIFA2',
        metric_maxSIFA=lambda d: d['Rec_Cat_short'] + '_maxSIFA2',
        metric_delta=lambda d: d['Rec_Cat_short'] + '_delta',
        metric_pctmet=lambda d: d['Rec_Cat_short'] + '_pct_met'
    )
)

long_metrics = pd.concat([
    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_selected','selected_area_ha']]
        .rename(columns={'metric_selected':'metric','selected_area_ha':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_minSIFA','min_SIFA2']]
        .rename(columns={'metric_minSIFA':'metric','min_SIFA2':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_maxSIFA','max_SIFA2']]
        .rename(columns={'metric_maxSIFA':'metric','max_SIFA2':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_delta','delta_to_target']]
        .rename(columns={'metric_delta':'metric','delta_to_target':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_pctmet','pct_met']]
        .rename(columns={'metric_pctmet':'metric','pct_met':'value'}),
], ignore_index=True)


wide_metrics = (
    long_metrics
    .pivot_table(
        index=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat'],
        columns='metric',
        values='value',
        aggfunc='first'
    )
    .reset_index()
)

wide_metrics['Rec_Cat'] = pd.to_numeric(wide_metrics['Rec_Cat'], errors='coerce').astype('Int64')

is_total = df['Rec_Cat'].astype(str).str.contains('Total', case=False, na=False)
targets_numeric = df.loc[~is_total].copy()
targets_total   = df.loc[ is_total].copy() 

targets_numeric['Rec_Cat'] = pd.to_numeric(targets_numeric['Rec_Cat'], errors='coerce').astype('Int64')
targets_numeric.rename(columns={'NAME':'WATER_MANAGEMENT_BASIN_NAME'}, inplace=True)

augmented_numeric = targets_numeric.merge(
    wide_metrics,
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat'],
    how='left'
)

augmented_full = pd.concat([augmented_numeric, targets_total], ignore_index=True)

orig_cols = ['Rec_Cat'] + [c for c in df.columns if c != 'Rec_Cat']
metric_cols = [c for c in augmented_full.columns if c not in orig_cols]
augmented_full = augmented_full[orig_cols + sorted(metric_cols)]
summary_counts.to_csv(os.path.join(table_outputs, "step_2a_summary_counts.csv"), index=False)
augmented_full.to_csv(os.path.join(table_outputs, "step_2a_summary.csv"), index=False)

In [ ]:
#step2b
ret_class_2b=return_gdf(sql_2b)
ret_class_2b['SIFA_2'] = pd.to_numeric(ret_class_2b['SIFA_2'], errors='coerce')
ret_class_2b['aflb_area_ha'] = pd.to_numeric(ret_class_2b['aflb_area_ha'], errors='coerce')
ret_class_2b = ret_class_2b.dropna(subset=['SIFA_2', 'aflb_area_ha'])
ret_class_2b['Rec_Cat_short'] = ret_class_2b['Rec_Cat_short'].astype(str).str.strip()
ret_class_2b['Rec_Cat'] = pd.to_numeric(ret_class_2b['Rec_Cat'], errors='coerce').astype('Int64')




In [ ]:
df = pd.read_csv(brwmb_targets_csv)
id_cols = ['NAME', 'Rec_Cat']
# short_cols = [c for c in df.columns if c not in id_cols]
short_cols = [c for c in df.columns if c.isalpha() and c not in id_cols]

targets_long = (
    df.melt(
        id_vars=id_cols,
        value_vars=short_cols,
        var_name='Rec_Cat_short',
        value_name='target_ha'
    )
    .assign(
        NAME=lambda d: d['NAME'].astype(str).str.strip(),
        Rec_Cat=lambda d: pd.to_numeric(d['Rec_Cat'], errors='coerce').astype('Int64'),
        Rec_Cat_short=lambda d: d['Rec_Cat_short'].astype(str).str.strip(),
        target_ha=lambda d: pd.to_numeric(d['target_ha'], errors='coerce')
    )
    .dropna(subset=['Rec_Cat'])
)

rules = {
    'Blueberry River': {
        2: ['MPM','LPC','LPM'],
        3: ['MPC','MPD'],
        4: ['HPC','HPD','HPM','LPD'],
    },
    'Lower Sikanni Chief River': {
        2: ['HPD','MPC','MPM'],
        3: ['HPM','MPD','LPC','LPD','LPM'],
        4: ['HPC'],
    },
    'Middle Beatton River': {
        3: ['LPC','LPD','LPM','MPC','MPD','MPM'],
        4: ['HPC','HPD','HPM'],
    },
    'Upper Beatton River': {
        3: ['LPC','LPD','LPM','MPC','MPD','MPM'],
        4: ['HPD'],
    },
}

rows = []
for basin, cats in rules.items():
    for rc, shorts in cats.items():
        for s in shorts:
            rows.append((basin, rc, s))

rules_df = pd.DataFrame(rows, columns=['NAME','Rec_Cat','Rec_Cat_short'])
rules_df['NAME'] = rules_df['NAME'].astype(str).str.strip()
rules_df['Rec_Cat'] = rules_df['Rec_Cat'].astype('Int64')
rules_df['Rec_Cat_short'] = rules_df['Rec_Cat_short'].astype(str).str.strip()

targets_2b = targets_long.merge(
    rules_df,
    on=['NAME','Rec_Cat','Rec_Cat_short'],
    how='inner'
)

targets_2b=targets_2b.rename(columns={'NAME':'WATER_MANAGEMENT_BASIN_NAME'})
gdf2 = ret_class_2b.merge(
    targets_2b[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha']],
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat', 'Rec_Cat_short'],
    how='inner'
)

selected = (
    gdf2
    .groupby(
        ['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'],
        group_keys=False,
        sort=False
    )
    .apply(lambda grp: pick_oldest_closest(grp, 'aflb_area_ha'))
    .reset_index(drop=True)
)

selection_gdf = selected[selected['picked']].copy()

total_counts = (
    gdf2
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'], as_index=False)
    .agg(total_polygons=('Rec_Cat', 'size'))
)

# Count USED polygons from picker
used_counts = (
    selection_gdf[selection_gdf['picked']]
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'], as_index=False)
    .agg(
        used_polygons=('picked', 'size'),
        selected_area_ha=('picked_area_sum', 'max'),
        delta_to_target=('delta_to_target', 'max')
    )
)

# Join totals + used
summary_counts = used_counts.merge(
    total_counts,
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
)
summary_counts=summary_counts[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha','selected_area_ha','delta_to_target','used_polygons','total_polygons']]
summary_counts=summary_counts.rename(columns={'Rec_Cat_short': 'for_rep'})
summary = (
    selection_gdf
    .groupby(
        ['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'],
        group_keys=False,
        sort=False
    )
    .agg(selected_area_ha=('aflb_area_ha', 'sum'),
         n_features=('aflb_area_ha', 'size'),
         max_SIFA2=('SIFA_2', 'max'),
         min_SIFA2=('SIFA_2', 'min'))
).reset_index()

summary['delta_to_target'] = summary['selected_area_ha'] - summary['target_ha']
summary['pct_met'] = (summary['selected_area_ha'] / summary['target_ha']) * 100.0

selection_gdf.to_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))

summary_wide = (
    summary
    .assign(
        metric_selected=lambda d: d['Rec_Cat_short'] + '_selected',
        metric_minSIFA=lambda d: d['Rec_Cat_short'] + '_minSIFA2',
        metric_maxSIFA=lambda d: d['Rec_Cat_short'] + '_maxSIFA2',
        metric_delta=lambda d: d['Rec_Cat_short'] + '_delta',
        metric_pctmet=lambda d: d['Rec_Cat_short'] + '_pct_met'
    )
)

long_metrics = pd.concat([
    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_selected','selected_area_ha']]
        .rename(columns={'metric_selected':'metric','selected_area_ha':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_minSIFA','min_SIFA2']]
        .rename(columns={'metric_minSIFA':'metric','min_SIFA2':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_maxSIFA','max_SIFA2']]
        .rename(columns={'metric_maxSIFA':'metric','max_SIFA2':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_delta','delta_to_target']]
        .rename(columns={'metric_delta':'metric','delta_to_target':'value'}),

    summary_wide[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','metric_pctmet','pct_met']]
        .rename(columns={'metric_pctmet':'metric','pct_met':'value'}),
], ignore_index=True)


wide_metrics = (
    long_metrics
    .pivot_table(
        index=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat'],
        columns='metric',
        values='value',
        aggfunc='first'
    )
    .reset_index()
)

wide_metrics['Rec_Cat'] = pd.to_numeric(wide_metrics['Rec_Cat'], errors='coerce').astype('Int64')

is_total = df['Rec_Cat'].astype(str).str.contains('Total', case=False, na=False)
targets_numeric = df.loc[~is_total].copy()
targets_total   = df.loc[ is_total].copy() 

targets_numeric['Rec_Cat'] = pd.to_numeric(targets_numeric['Rec_Cat'], errors='coerce').astype('Int64')
targets_numeric.rename(columns={'NAME':'WATER_MANAGEMENT_BASIN_NAME'}, inplace=True)

augmented_numeric = targets_numeric.merge(
    wide_metrics,
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat'],
    how='left'
)

augmented_full = pd.concat([augmented_numeric, targets_total], ignore_index=True)

orig_cols = ['Rec_Cat'] + [c for c in df.columns if c != 'Rec_Cat']
metric_cols = [c for c in augmented_full.columns if c not in orig_cols]
augmented_full = augmented_full[orig_cols + sorted(metric_cols)]
augmented_full.to_csv(os.path.join(table_outputs, "step_2b_summary.csv"), index=False)
summary_counts.to_csv(os.path.join(table_outputs, "step_2b_summary_counts.csv"), index=False)


#setp 2c 

select by location intersect, step 2b, with CIA dissolve and TSU_WMB, multipart to single part, then remove stands that intersect with HV1 DZs 

In [ ]:
# cia_ciap=gpd.read_parquet(cia_dissolve)
# hv1_dv=gpd.read_parquet(os.path.join(input_parquet,'hv1.parquet'),filters=[('Zone', '==', 'Development')])
# tsu=gpd.read_parquet(os.path.join(input_parquet,'tsu_by_wmb.parquet'))
# selected=gpd.read_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))

In [ ]:
#data set now to large switching it manual 
# #merge cia and tsu
# cia_tsu= gpd.overlay(cia_ciap, tsu, how='union')
# info(f"After merging CIA_CIAP_noHV1TU and trapline_clip, cia_tsu has {len(cia_tsu)} features")
# # select all stands that intersect with cia_tsu
# ret_class_2c=gpd.overlay(selected, cia_tsu, how='intersection')
# info(f"After intersecting selected with cia_tsu, ret_class_2c has {len(ret_class_2c)} features")
# #remove features that intersect with hv1
# ret_class_2c_nohv1=gpd.overlay(ret_class_2c, hv1_dv, how='difference')
# info(f"After removing features that intersect with HV1_Clip, ret_class_2c_nohv1 has {len(ret_class_2c_nohv1)} features")
# ret_class_2c_nohv1.to_parquet(os.path.join(output_dir, "ret_class_2c_nohv1.parquet"))

# ret_class_2c_nohv1=gpd.overlay(selected, hv1_dv, how='difference')
# select_in=pd.concat([cia_ciap,tsu], ignore_index=True)
# select_in=select_in.dissolve()
# ret_class_2c_nohv1=gpd.overlay(ret_class_2c_nohv1, select_in, how='intersection')
# ret_class_2c_nohv1.to_parquet(os.path.join(output_dir, "ret_class_2c_nohv1.parquet"))

In [ ]:

# ret_class_2c_nohv1=gpd.overlay(ret_class_2c_nohv1, select_in, how='intersection')


In [ ]:
# #step 2c: remove any remaining hv1 and keep only inside CIA/TSU
# def snap_and_validate(gdf, grid=0.5):
#     """Snap coords to a grid (meters in EPSG:3005) and make valid to reduce overlay complexity."""
#     geoms = gdf.geometry.values
#     geoms = make_valid(geoms)                   
#     geoms = set_precision(geoms, grid)         
#     gdf = gdf.set_geometry(geoms)
#     return gdf

# grid_size = 0.5  
# selected  = snap_and_validate(selected,  grid=grid_size)
# hv1_dv    = snap_and_validate(hv1_dv,    grid=grid_size)

# select_in = gpd.GeoDataFrame(
#     geometry=pd.concat([cia_ciap[['geometry']], tsu[['geometry']]], ignore_index=True)['geometry'],
#     crs=selected.crs
# )
# select_in = snap_and_validate(select_in, grid=grid_size)
# aoi_union = unary_union(select_in.geometry.values)   

# aoi_bbox = box(*gpd.GeoSeries([aoi_union], crs=selected.crs).total_bounds)
# sel_idx = selected.sindex.query(aoi_bbox, predicate='intersects')
# selected_small = selected.iloc[sel_idx].copy()

# hv1_idx = hv1_dv.sindex.query(aoi_bbox, predicate='intersects')
# hv1_small = hv1_dv.iloc[hv1_idx].copy()

# hv1_union = unary_union(hv1_small.geometry.values)

# diff_geoms = shapely.difference(selected_small.geometry.values, hv1_union)
# selected_small = selected_small.set_geometry(diff_geoms)
# selected_small = selected_small.loc[~selected_small.geometry.is_empty]

# clip_geoms = shapely.intersection(selected_small.geometry.values, aoi_union)
# selected_small = selected_small.set_geometry(clip_geoms)
# selected_small = selected_small.loc[~selected_small.geometry.is_empty]

# selected_small = selected_small[selected_small.geometry.geom_type.isin(['Polygon','MultiPolygon'])]

# out_path = os.path.join(output_dir, "ret_class_2c_nohv1.parquet")
# selected_small.to_parquet(out_path)


In [ ]:
ret = gpd.read_parquet(os.path.join(output_dir,os.path.join(output_dir, "ret_class_2c_intermidate.parquet")))
# if 'ret_class_2c_nohv1' not in globals():
#     ret = gpd.read_parquet(os.path.join(output_dir,os.path.join(output_dir, "ret_class_2c_nohv1_aflb.parquet")))
# else:
#     try:
#         ret=ret_class_2c_nohv1
#     except:
#         error("ret_class_2c_nohv1 not found in globals and failed to read from parquet. Please check the file path and variable name.")

                     
# ret=ret.dropna(subset=['uid'])
# info(f"After dropping records without uid, ret has {len(ret)} features")

# ret_dissolved = ret.dissolve(by='uid', aggfunc={'aflb_area_ha': 'sum'})
# ret_dissolved = ret_dissolved.reset_index()
# info(f"Dissolved ret by uid, now has {len(ret_dissolved)} features with summed aflb_area by uid")

# ret=gpd.sjoin(ret_dissolved, ret_class_2c_nohv1, how='left', predicate='crosses')
# info(f'of features in sjoin {len(ret)}')
cols_to_drop = [
    c for c in ret.columns
    if c.endswith('_2') and c != 'SIFA_2'
]

ret = ret.drop(columns=cols_to_drop)


if 'Ret_Cat' not in ret.columns:
    ret = ret.rename(columns={'Rec_Cat_1': 'Rec_Cat'})

if 'Rec_Cat_short' not in ret.columns:
    ret = ret.rename(columns={'Rec_Cat_short_1': 'Rec_Cat_short'})
    
cols = [c for c in ret.columns if "aflb_area" in c]
if cols:
    ret = ret.drop(columns=cols)
if ret.crs is None or ret.crs.to_epsg() != 3005:
    ret = ret.to_crs(3005)
ret["area_ha"] = ret.geometry.area / 10000.0
ret["aflb_fact"] = pd.to_numeric(ret["aflb_fact"], errors="coerce").fillna(0)
ret["aflb_area_ha"] = ret["aflb_fact"] * ret["area_ha"]
ret2=ret.copy()
#inputs gdf to calculate on, targets df (targets_2b), and output csv path for summary table with metrics comparing to targets
ret2['SIFA_2'] = pd.to_numeric(ret2['SIFA_2'], errors='coerce')
ret2['aflb_area_ha'] = pd.to_numeric(ret2['aflb_area_ha'], errors='coerce')
ret2 = ret2.dropna(subset=['SIFA_2', 'aflb_area_ha'])
ret2['Rec_Cat_short'] = ret2['Rec_Cat_short'].astype(str).str.strip()
ret2['Rec_Cat'] = pd.to_numeric(ret2['Rec_Cat'], errors='coerce').astype('Int64')

In [ ]:
gdf2 = ret2.merge(
    targets_2b[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha']],
    on=['WATER_MANAGEMENT_BASIN_NAME', 'Rec_Cat', 'Rec_Cat_short'],
    how='inner'
)

if 'target_ha' not in gdf2.columns:
    gdf2 = gdf2.rename(columns={'target_ha_y': 'target_ha'})
    
    
selected = (
    gdf2
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat', 'Rec_Cat_short', 'target_ha'], group_keys=False, sort=False)
    .apply(lambda grp: pick_oldest_closest(grp, 'aflb_area_ha'))
    .reset_index(drop=True)
    
)

selection_gdf = selected[selected['picked']].copy()

selected = (
    gdf2
    .groupby(
        ['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'],
        group_keys=False,
        sort=False
    )
    .apply(lambda grp: pick_oldest_closest(grp, 'aflb_area_ha'))
    .reset_index(drop=True)
)

selection_gdf = selected[selected['picked']].copy()


    
total_counts = (
    gdf2
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'], as_index=False)
    .agg(total_polygons=('Rec_Cat', 'size'))
)

# Count USED polygons from picker
used_counts = (
    selection_gdf[selection_gdf['picked']]
    .groupby(['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha'], as_index=False)
    .agg(
        used_polygons=('picked', 'size'),
        selected_area_ha=('picked_area_sum', 'max'),
        delta_to_target=('delta_to_target', 'max')
    )
)

# Join totals + used
summary_counts = used_counts.merge(
    total_counts,
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'],
    how='left'
)

summary_counts=summary_counts[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha','selected_area_ha','delta_to_target','used_polygons']]
summary_counts=summary_counts.rename(columns={'Rec_Cat_short': 'for_rep'})
summary_counts.to_csv(os.path.join(table_outputs, "step_2c_summary_counts.csv"), index=False)

selection_gdf = gpd.GeoDataFrame(
    selection_gdf,
    geometry="geometry",
    crs=selection_gdf.crs
)

selection_gdf.to_parquet(os.path.join(output_dir, "ret_class_2c_nohv1.parquet"))

In [ ]:
ret_class_2a=gpd.read_parquet(os.path.join(output_dir, "ret_class_2a.parquet"))
selection_gdf=gpd.read_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))


In [ ]:
ret_class_2c_nohv1=selection_gdf

In [ ]:
if "ret_class_2a" not in globals():
    info('read 2a')
    ret_class_2a=gpd.read_parquet(os.path.join(output_dir, "ret_class_2a.parquet"))
if "selection_gdf" not in globals():
    info('read 2b')
    selection_gdf=gpd.read_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))
if"ret_class_2c_nohv1" not in globals():
    info('read 2c')
    ret_class_2c_nohv1=gpd.read_parquet(os.path.join(output_dir, "ret_class_2c_nohv1.parquet"))
if "targets_long" not in globals():
    targets=pd.read_csv(brwmb_targets_csv)
    targets_long = (
    targets.melt(
        id_vars=id_cols,
        value_vars=short_cols,
        var_name='Rec_Cat_short',
        value_name='target_ha'
    )
    .assign(
        NAME=lambda d: d['NAME'].astype(str).str.strip(),
        Rec_Cat=lambda d: pd.to_numeric(d['Rec_Cat'], errors='coerce').astype('Int64'),
        Rec_Cat_short=lambda d: d['Rec_Cat_short'].astype(str).str.strip(),
        target_ha=lambda d: pd.to_numeric(d['target_ha'], errors='coerce')
    )
    .dropna(subset=['Rec_Cat'])
    )

    rules = {
        'Blueberry River': {
            2: ['MPM','LPC','LPM'],
            3: ['MPC','MPD'],
            4: ['HPC','HPD','HPM','LPD'],
        },
        'Lower Sikanni Chief River': {
            2: ['HPD','MPC','MPM'],
            3: ['HPM','MPD','LPC','LPD','LPM'],
            4: ['HPC'],
        },
        'Middle Beatton River': {
            3: ['LPC','LPD','LPM','MPC','MPD','MPM'],
            4: ['HPC','HPD','HPM'],
        },
        'Upper Beatton River': {
            3: ['LPC','LPD','LPM','MPC','MPD','MPM'],
            4: ['HPD'],
        },
    }
targets_long=targets_long.rename(columns={'NAME':'WATER_MANAGEMENT_BASIN_NAME'})

    # targets_long['Rec_Cat'] = targets_long['Rec_Cat'].astype('Int64')
    # targets_long['Rec_Cat_short'] = targets_long['Rec_Cat_short'].astype(str).str.strip()
    # targets_long['target_ha'] = pd.to_numeric(targets_long['target_ha'], errors='coerce').fillna(0)
    
    
def normalize_gdf_for_summary(geodf):
    try:
        geodf['Rec_Cat'] = pd.to_numeric(geodf['Rec_Cat'], errors='coerce').astype('Int64')
    except:
        geodf['Rec_Cat_1'] = pd.to_numeric(geodf['Rec_Cat_1'], errors='coerce').astype('Int64')
        geodf.rename(columns={'Rec_Cat_1': 'Rec_Cat'}, inplace=True)
        
    geodf['Rec_Cat_short'] = geodf['Rec_Cat_short'].astype(str).str.strip().str.upper()
    geodf['aflb_fact'] = pd.to_numeric(geodf['aflb_fact'], errors='coerce')
    geodf['area_ha_geom'] = geodf.geometry.area / 10_000.0
    geodf['aflb_area'] = geodf['area_ha_geom'] * geodf['aflb_fact']
    geodf = geodf.loc[geodf['aflb_area'].notna() & (geodf['aflb_area'] > 0)].copy()
    return geodf

ret_class_2a= normalize_gdf_for_summary(ret_class_2a)
selection_gdf=normalize_gdf_for_summary(selection_gdf)
ret_class_2c_nohv1=normalize_gdf_for_summary(ret_class_2c_nohv1)

pool = pd.concat([ret_class_2a, selection_gdf], ignore_index=True)

aflb_totals = (pool
               .groupby(['WATER_MANAGEMENT_BASIN_NAME',"Rec_Cat", "Rec_Cat_short"], dropna=False)["aflb_area"]
               .sum()
               .rename("Step 2a/b AFLB Area(ha)")
               .reset_index())

summary = (
    targets_long
    .merge(aflb_totals, on=['WATER_MANAGEMENT_BASIN_NAME',"Rec_Cat", "Rec_Cat_short"], how="left")
    .fillna({"Step 2a/b AFLB Area(ha)": 0.0})
)    

aflb_totals_2c=(ret_class_2c_nohv1
            .groupby(['WATER_MANAGEMENT_BASIN_NAME',"Rec_Cat", "Rec_Cat_short"], dropna=False)["aflb_area"]
               .sum()
               .rename("Step 2c AFLB Area(ha)")
               .reset_index())

summary = (
        summary
        .merge(aflb_totals_2c, on=['WATER_MANAGEMENT_BASIN_NAME',"Rec_Cat", "Rec_Cat_short"], how="left")
        .fillna({"accum_aflb_ha": 0.0})
)
summary.rename(columns={'Rec_Cat':'Rec Cat',
                        'Rec_Cat_short': 'For Rep',
                        'target_ha':'Target AFLB(ha)'
                        }, inplace=True)
summary.to_excel(os.path.join(table_outputs,'step_2a_b_c_targets.xlsx'))


In [ ]:
# keeps crashing doing it manually 
# Short Fall Layer
# - using only for orange categories so start with selection 
# # - create priority p1-crosses/in CIA and TSU, p2 TSU only, p3 CIA only  
# # - cumulative total for each 

# sql_2a=f"""
#     SELECT *,ST_AsText(geometry) as wkt, from '{ret_class}'
#     WHERE
#   (WATER_MANAGEMENT_BASIN_NAME = 'Blueberry River'
#         AND(
#   	Rec_Cat_short='HPC' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPD' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPM' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='MPC' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPM' and Rec_Cat in (2,3,4,5) OR
# 	Rec_Cat_short='LPC' and Rec_Cat in (2,3,4,5) OR
# 	Rec_Cat_short='LPD' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='LPM' and Rec_Cat in (2,3,4,5)))
	
# OR (WATER_MANAGEMENT_BASIN_NAME = 'Lower Sikanni Chief River'
# AND(
#   	Rec_Cat_short='HPC' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPD' and Rec_Cat in (2,3,4,5) OR
# 	Rec_Cat_short='HPM' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPC' and Rec_Cat in (2,3,4,5) OR
# 	Rec_Cat_short='MPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPM' and Rec_Cat in (2,3,4,5) OR
# 	Rec_Cat_short='LPC' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPM' and Rec_Cat in (3,4,5)))

# OR (WATER_MANAGEMENT_BASIN_NAME = 'Middle Beatton River'
# AND ( 
#   	Rec_Cat_short='HPC' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPD' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPM' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='MPC' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPM' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPC' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPM' and Rec_Cat in (3,4,5)))	
	
# OR (WATER_MANAGEMENT_BASIN_NAME = 'Upper Beatton River'
# AND (
#   	Rec_Cat_short='HPC' and Rec_Cat in (5) OR
# 	Rec_Cat_short='HPD' and Rec_Cat in (4,5) OR
# 	Rec_Cat_short='HPM' and Rec_Cat in (5) OR
# 	Rec_Cat_short='MPC' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='MPM' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPC' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPD' and Rec_Cat in (3,4,5) OR
# 	Rec_Cat_short='LPM' and Rec_Cat in (3,4,5)))	
# #     """
# #return query results as gdf
# step_2d_base=return_gdf(sql_2a)

# - remove stands that overlap/crosses with step2b 
# step2b_selection=gpd.read_parquet(os.path.join(output_dir, "ret_class_2b.parquet"))
# gpd intersect difference 
# step_2d_no_step2b=gpd.overlay(step_2d_base, step2b_selection, how='difference')
# - remove HV1 DZ
# step_2d_no_step2b=gpd.overlay(step_2d_no_step2b, hv1_dv, how='difference')

# priority_2d


# Step 2d
manual portion, 
staring with rec_class_all
    - verify protection_2d is populated
    - remove stands that overlap with step 2b(ret_class_2b.parquet), select by location within (invert selection)
    - remove stands that overlap with HV1 DZ , select by location interesct, remove from current selection 
    - save as step_2d_base_aflb.parquet

In [ ]:
#read in manual 2d base class 
step_2d_base_gdf=gpd.read_parquet(step2d_base)

summary_2c=pd.read_csv(os.path.join(table_outputs, "step_2c_summary_counts.csv"))
dtt=summary_2c['delta_to_target']

# Negative delta_to_target = shortfalls
shortfalls = summary_2c[summary_2c["delta_to_target"] < 0].copy()

# Convert negative deltas into positive target amounts for 2d
shortfalls["target_ha"] = -shortfalls["delta_to_target"]

# Keep only columns the allocation function needs
targets_2d = shortfalls[[
    "WATER_MANAGEMENT_BASIN_NAME",
    "Rec_Cat",
    "for_rep",
    "target_ha"
]].copy()

# Normalize case for Rec_Cat_short (required by allocator)
targets_2d["for_rep"] = (
    targets_2d["for_rep"].astype(str).str.upper().str.strip()
)

In [ ]:

def allocate_representation_2d(
    gdf: gpd.GeoDataFrame,
    targets_df: pd.DataFrame,
    *,
    basin_col: str = "WATER_MANAGEMENT_BASIN_NAME",
    area_col: str = "aflb_area_ha",       
    rec_cat_col: str = "Rec_Cat",
    rec_short_col: str = "Rec_Cat_short",
    prio_col: int = "priority_2d",
    age_col: str = "SIFA_2",
    max_rec_cat: int | None = None,        # default: infer from data
    prio_order: Tuple[int, ...] = (1, 2, 3),
    backfill_mode: str = "up",             # "up" | "down" | "bidirectional"
    protect_order: bool = True,
    overshoot_mode: str = "closest"        # "closest" | "prefer_under" | "prefer_over"
) -> Tuple[gpd.GeoDataFrame, gpd.GeoDataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Greedy allocation of whole stands to meet Rec_Cat + Rec_Cat_short targets, by priority (1->3),
    oldest-first, with backfill across Rec_Cat as configured.

    Robustness:
      - Excludes candidates with NaN/<=0 area to prevent NaN propagation.
      - Uses np.nansum for area aggregations.
      - Overshoot handling selects the option closest to target (under or over), with tie-breakers:
          1) prefer more area from the target Rec_Cat,
          2) higher priority (1>2>3),
          3) older age (higher SIFA_2).

    Summary output:
      - One row per target.
      - Includes ha_in_rcat_{SRC}_p{P} columns for each source Rec_Cat in [1..max_rec_cat]
        and each priority in prio_order.
    """
    # --- Required columns check
    required_cols = {area_col, rec_cat_col, rec_short_col, prio_col, age_col, basin_col}
    missing = required_cols - set(gdf.columns)
    if missing:
        raise ValueError(f"gdf is missing required columns: {sorted(missing)}")
    
    
    required_targets = {rec_cat_col, rec_short_col, "target_ha", basin_col}
    missing_targets = required_targets - set(targets_df.columns)
    if missing_targets:
        raise ValueError(f"targets_df is missing required columns: {sorted(missing_targets)}")


    if overshoot_mode not in {"closest", "prefer_under", "prefer_over"}:
        raise ValueError("overshoot_mode must be one of {'closest','prefer_under','prefer_over'}")

    # --- Copy + normalize types
    gdf_out = gdf.copy()
    gdf_out["_allocated"] = False
    gdf_out["cat_class_2d"] = pd.Series([np.nan] * len(gdf_out), dtype="object")

    # Normalize case/types
    gdf_out[rec_short_col] = gdf_out[rec_short_col].astype(str).str.upper()
    targets_df = targets_df.copy()
    targets_df[rec_short_col] = targets_df[rec_short_col].astype(str).str.upper()
    
    # Basin normalization
    gdf_out[basin_col] = gdf_out[basin_col].astype(str).str.strip()
    targets_df[basin_col] = targets_df[basin_col].astype(str).str.strip()

    # Coerce numeric area/age
    gdf_out[area_col] = pd.to_numeric(gdf_out[area_col], errors="coerce")
    gdf_out[age_col] = pd.to_numeric(gdf_out[age_col], errors="coerce")
    gdf_out[rec_cat_col] = pd.to_numeric(gdf_out[rec_cat_col], errors="coerce").astype("Int64")
    gdf_out[prio_col] = pd.to_numeric(gdf_out[prio_col], errors="coerce").astype("Int64")
    targets_df[rec_cat_col] = pd.to_numeric(targets_df[rec_cat_col], errors="coerce").astype(int)
    targets_df["target_ha"] = pd.to_numeric(targets_df["target_ha"], errors="coerce").astype(float)

    # Infer max_rec_cat if not provided
    if max_rec_cat is None:
        max_rec_cat = int(np.nanmax(gdf_out[rec_cat_col].dropna()))

    # Valid area mask (use throughout)
    valid_area_mask = gdf_out[area_col].notna() & (gdf_out[area_col] > 0)

    # Backfill sequence helper
    def backfill_sequence(start: int) -> List[int]:
        if backfill_mode == "up":
            return [rc for rc in range(start, max_rec_cat + 1)]
        elif backfill_mode == "down":
            return [rc for rc in range(start, 0, -1)]
        elif backfill_mode == "bidirectional":
            seq = [start]
            k = 1
            while True:
                candidates = []
                up = start + k
                down = start - k
                if 1 <= up <= max_rec_cat:
                    candidates.append(up)
                if 1 <= down <= max_rec_cat:
                    candidates.append(down)
                if not candidates:
                    break
                seq.extend(candidates)
                k += 1
            # de-dup in order
            seen = set()
            out = []
            for x in seq:
                if x not in seen:
                    out.append(x)
                    seen.add(x)
            return out
        else:
            raise ValueError("Invalid backfill_mode. Use 'up', 'down', or 'bidirectional'.")

    allocation_rows: List[Dict] = []
    summary_rows: List[Dict] = []

    t_iter = targets_df.itertuples(index=False) if protect_order else \
             targets_df.sort_values([basin_col,rec_cat_col, rec_short_col]).itertuples(index=False)

    # Helper: overshoot set preference
    def prefer_set(contrib_all: List[Dict], contrib_drop_last: List[Dict],
                   tgt_rec_cat: int, tgt_area: float,
                   err_all: float, err_drop: float) -> str:
        if err_drop < err_all:
            return "drop"
        if err_all < err_drop:
            return "all"

        # Tie: overshoot policy
        achieved_all = float(np.nansum([c["area"] for c in contrib_all]))
        achieved_drop = float(np.nansum([c["area"] for c in contrib_drop_last]))
        under_all = achieved_all < tgt_area
        under_drop = achieved_drop < tgt_area

        if overshoot_mode == "prefer_under" and (under_all != under_drop):
            return "all" if under_all else "drop"
        if overshoot_mode == "prefer_over" and (under_all != under_drop):
            return "drop" if under_all else "all"

        # Domain tie-breakers
        def score(contrib: List[Dict]) -> tuple:
            area_in_target = float(np.nansum([c["area"] for c in contrib if c["source_rec_cat"] == tgt_rec_cat]))
            prio_weights = {1: 3, 2: 2, 3: 1}
            prio_score = float(np.nansum([c["area"] * prio_weights.get(c["source_priority"], 0) for c in contrib]))
            ages = [c["age"] for c in contrib if pd.notna(c["age"])]
            age_score = float(np.nanmean(ages)) if ages else 0.0
            return (area_in_target, prio_score, age_score)

        return "all" if score(contrib_all) >= score(contrib_drop_last) else "drop"

    for t in t_iter:
        tgt_basin = getattr(t, basin_col) 
        tgt_rec_cat = int(getattr(t, rec_cat_col))
        tgt_short = getattr(t, rec_short_col)
        tgt_area = float(getattr(t, "target_ha"))
        label = f"{tgt_rec_cat}{tgt_short}"

        selected_indices: List[int] = []
        contrib: List[Dict] = []
        cum_area = 0.0

        for rc in backfill_sequence(tgt_rec_cat):
            for p in prio_order:
                mask = (
                    (~gdf_out["_allocated"])
                    & valid_area_mask
                    &(gdf_out[basin_col] == tgt_basin) 
                    &(gdf_out[rec_short_col] == tgt_short)  
                    &(gdf_out[rec_cat_col] == rc) 
                    &(gdf_out[prio_col] == p)
                )
                if not mask.any():
                    continue

                pool = gdf_out.loc[mask].sort_values(by=age_col, ascending=False)

                for idx, row in pool.iterrows():
                    a = float(row[area_col]) if pd.notna(row[area_col]) else 0.0
                    if not np.isfinite(a) or a <= 0:
                        continue

                    selected_indices.append(idx)
                    contrib.append({
                        "target_basin": tgt_basin,
                        "target_rec_cat": tgt_rec_cat,
                        "target_short": tgt_short,
                        "source_rec_cat": rc,
                        "source_priority": p,
                        "area": a,
                        "age": float(row[age_col]) if pd.notna(row[age_col]) else np.nan,
                        "idx": idx
                    })
                    gdf_out.at[idx, "_allocated"] = True
                    cum_area += a

                    if cum_area >= tgt_area:
                        break
                if cum_area >= tgt_area:
                    break
            if cum_area >= tgt_area:
                break

        # Overshoot resolution
        if selected_indices:
            achieved_all = float(np.nansum([c["area"] for c in contrib]))
            err_all = abs(achieved_all - tgt_area)
            if len(contrib) >= 1:
                last_idx = selected_indices[-1]
                contrib_drop = contrib[:-1]
                err_drop = abs(float(np.nansum([c["area"] for c in contrib_drop])) - tgt_area)
                if prefer_set(contrib, contrib_drop, tgt_rec_cat, tgt_area, err_all, err_drop) == "drop":
                    gdf_out.at[last_idx, "_allocated"] = False
                    selected_indices.pop()
                    contrib = contrib_drop
                    cum_area = float(np.nansum([c["area"] for c in contrib]))

        # Label selected with the TARGET cat_class_2d
        if selected_indices:
            gdf_out.loc[selected_indices, "cat_class_2d"] = label

        # Allocation log rows
        for c in contrib:
            allocation_rows.append({
                "target_basin": tgt_basin,
                "target_cat_class_2d": label,
                "target_rec_cat": tgt_rec_cat,
                "target_rec_short": tgt_short,
                "source_rec_cat": c["source_rec_cat"],
                "source_priority": c["source_priority"],
                "aflb_area_ha": c["area"],
                "age": c["age"],
                "gdf_index": c["idx"]
            })

        # --- Per-target summary (wide by source Rec_Cat × priority) ---
        def nsum(iterable):
            arr = np.array(list(iterable), dtype="float64")
            return float(np.nansum(arr)) if arr.size else 0.0

        achieved_ha = nsum(c["area"] for c in contrib)
        ages = [c["age"] for c in contrib if pd.notna(c["age"])]
        min_age = float(np.min(ages)) if len(ages) else np.nan
        max_age = float(np.max(ages)) if len(ages) else np.nan

        # Build the wide columns programmatically
        src_rcat_prio_cols: Dict[str, float] = {}
        for src_rc in range(1, max_rec_cat + 1):
            for p in prio_order:
                key = f"ha_in_rcat_{src_rc}_p{p}"
                val = nsum(c["area"] for c in contrib
                           if c["source_rec_cat"] == src_rc and c["source_priority"] == p)
                src_rcat_prio_cols[key] = round(val, 5)

        summary_rows.append({
            "target_basin": tgt_basin, 
            "target_rec_cat": tgt_rec_cat,
            "target_rec_short": tgt_short,
            "target_ha": tgt_area,
            "achieved_ha": round(achieved_ha, 5),
            "delta_ha": round(achieved_ha - tgt_area, 5),
            "abs_delta_ha": round(abs(achieved_ha - tgt_area), 5),
            "min_SIFA_2": min_age,
            "max_SIFA_2": max_age,
            **src_rcat_prio_cols,   # <-- new wide matrix
        })

    allocation_log = pd.DataFrame(allocation_rows)
    summary_per_target = pd.DataFrame(summary_rows)
    selected_gdf = gdf_out[gdf_out["cat_class_2d"].notna()].copy()

    return gdf_out.drop(columns=["_allocated"]), selected_gdf, summary_per_target, allocation_log


In [ ]:
# 1) Load your stands (only columns we need -> faster)
use_cols = ["Rec_Cat", "Rec_Cat_short", "priority_2d", "SIFA_2", "aflb_area_ha", "geometry"]
try:
    targets_2d.rename(columns={'for_rep':'Rec_Cat_short'}, inplace=True)
except:
    pass 

# 3) Run allocation (default backfill “up”: N -> N+1 -> ... -> 5)
gdf_out, selected_gdf, summary_df, log_df = allocate_representation_2d(
    step_2d_base_gdf,
    targets_2d,
    basin_col= "WATER_MANAGEMENT_BASIN_NAME",
    area_col='aflb_area_ha',         
    rec_cat_col="Rec_Cat",
    rec_short_col="Rec_Cat_short",
    prio_col="priority_2d",
    age_col="SIFA_2",
    backfill_mode="up"           # per your example (e.g., 2 -> 3 -> 4 -> 5)
)

# 4) Persist results if needed
selected_gdf.to_parquet(os.path.join(output_dir, "ret_class_2d.parquet"))
summary_df.to_excel(os.path.join(table_outputs, "step_2d_summary_by_target.xlsx"), index=False)
log_df.to_csv(os.path.join(output_dir, "2d_allocation_log.csv"), index=False)

In [ ]:
# step 2e merge layers from 2 a,c,d 
#read 
step_2a_output=gpd.read_parquet(os.path.join(output_dir, "ret_class_2a.parquet")) 
info(len(step_2a_output))
step_2c_output=gpd.read_parquet(os.path.join(output_dir, "ret_class_2c_nohv1.parquet"))
info(len(step_2c_output))
step_2d_output=gpd.read_parquet(os.path.join(output_dir, "ret_class_2d.parquet"))
info(len(step_2d_output))

In [ ]:
#again remove any duplicate cols that end with _2 except for SIFA_2
def drop_duplicate_cols(gdf: gpd.GeoDataFrame, suffix: str = "_2", keep_col: str = "SIFA_2") -> gpd.GeoDataFrame:
    cols_to_drop = [c for c in gdf.columns if c.endswith(suffix) and c != keep_col]
    return gdf.drop(columns=cols_to_drop)
step_2a_clean = drop_duplicate_cols(step_2a_output)
step_2c_clean = drop_duplicate_cols(step_2c_output)
step_2d_clean = drop_duplicate_cols(step_2d_output)

In [ ]:
# merge gdfs
step_2e_all = pd.concat([step_2a_clean, step_2c_clean, step_2d_clean], ignore_index=True)
step_2e_all['aflb_area_ha']=step_2e_all['aflb_fact']*(step_2e_all.geometry.area/10000.0)
info(f"After merging step 2a, 2c, and 2d, step_2e_all has {len(step_2e_all)} features")
step_2e_all.to_parquet(os.path.join(output_dir, "ret_class_2e_all.parquet"))

In [ ]:
df = pd.read_csv(brwmb_targets_csv)
id_cols = ['NAME', 'Rec_Cat']
# short_cols = [c for c in df.columns if c not in id_cols]
short_cols = [c for c in df.columns if c.isalpha() and c not in id_cols]

targets_long = (
    df.melt(
        id_vars=id_cols,
        value_vars=short_cols,
        var_name='Rec_Cat_short',
        value_name='target_ha'
    )
    .assign(
        NAME=lambda d: d['NAME'].astype(str).str.strip(),
        Rec_Cat=lambda d: pd.to_numeric(d['Rec_Cat'], errors='coerce').astype('Int64'),
        Rec_Cat_short=lambda d: d['Rec_Cat_short'].astype(str).str.strip(),
        target_ha=lambda d: pd.to_numeric(d['target_ha'], errors='coerce')
    )
    .dropna(subset=['Rec_Cat'])
)
targets_long.rename(columns={'NAME':'WATER_MANAGEMENT_BASIN_NAME'}, inplace=True)

In [ ]:
summary = (
    step_2e_all
    .groupby(
        ['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'],
        group_keys=False,
        sort=False
    )
    .agg(selected_area_ha=('aflb_area_ha', 'sum'),
         n_features=('aflb_area_ha', 'size'),
         max_SIFA2=('SIFA_2', 'max'),
         min_SIFA2=('SIFA_2', 'min'))
).reset_index()


In [ ]:
summary2=summary.merge(targets_long, on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'], how='inner')
summary2['delta_to_target'] = summary2['selected_area_ha'] - summary2['target_ha']

summary2=summary2[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha','selected_area_ha','delta_to_target','n_features','max_SIFA2','min_SIFA2']]
summary2=summary2.rename(columns={'Rec_Cat_short': 'for_rep'})
summary2.to_csv(os.path.join(table_outputs, "step_2e_summary.csv"), index=False)

In [ ]:
#total targets
step_2ac_all = pd.concat([step_2a_clean, step_2c_clean]) 
summary = (
    step_2ac_all
    .groupby(
        ['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'],
        group_keys=False,
        sort=False
    )
    .agg(selected_area_ha=('aflb_area_ha', 'sum'),
         n_features=('aflb_area_ha', 'size'),
         max_SIFA2=('SIFA_2', 'max'),
         min_SIFA2=('SIFA_2', 'min'))
).reset_index()

In [ ]:
summary_2d=pd.read_excel(os.path.join(table_outputs, "step_2d_summary_by_target.xlsx"))
summary_2d=summary_2d[['target_basin','target_rec_cat','target_rec_short','achieved_ha']]
summary_2d = summary_2d.rename(columns={
    'target_basin': 'WATER_MANAGEMENT_BASIN_NAME',
    'target_rec_cat': 'Rec_Cat',
    'target_rec_short': 'Rec_Cat_short'
})

summary2 = summary.merge(
    summary_2d,
    on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'],
    how='outer'
)
summary2['selected_area_total_ha'] = (
    summary2['selected_area_ha'] + summary2['achieved_ha'].fillna(0)
)

summary2['achieved_ha'] = summary2['achieved_ha'].fillna(0)




In [ ]:
summary2=summary2.merge(targets_long, on=['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short'], how='inner')
summary2['delta_to_target'] = summary2['selected_area_total_ha'] - summary2['target_ha']
summary2=summary2[['WATER_MANAGEMENT_BASIN_NAME','Rec_Cat','Rec_Cat_short','target_ha','selected_area_ha','achieved_ha','selected_area_total_ha','delta_to_target','max_SIFA2','min_SIFA2']]
summary2=summary2.rename(columns={'Rec_Cat_short': 'for_rep', 'selected_area_ha': '2 a/b selected_area_ha', 'achieved_ha': '2d achieved_ha', 'selected_area_total_ha': '2a/b + 2d total selected area (ha)'})
summary2.to_excel(os.path.join(table_outputs, "step_2_total_summary.xlsx"), index=False)
